# Decepticons Price Prediction
This combines the decepticons transformer for `review_score_value` prediction
and the linear regerssion head for the actual `price` prediction.

## The Prediction procedure

In [ ]:
import warnings
import numpy as np
from urllib3.exceptions import NotOpenSSLWarning
import torch
warnings.filterwarnings("ignore", category=NotOpenSSLWarning)
from torch.utils.data import DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import decepticons as transformer
from decepticon_regressor import decepticons_regressor as regressor
from decepticons import decepticons as transformer
from decepticons import ReviewDataset as ReviewDataset

class decepticons_price_prediction:

    def __init__(self):
        self.predicted_review_score_values = []
        self.predicted_prices = []
        self.regressor = regressor()  # Instantiate the class
        self.regressor.load_model()
        self.transformer = transformer()

    def predict_prices(self):
        self.transformer = transformer()
        train_df, val_df, test_df = self.transformer.split_data()
        print(test_df)

        train_labels, val_labels, test_labels = self.transformer.discretize_labels(
            train_df=train_df,
            val_df=val_df,
            test_df=test_df,
        )

        tokens_train, tokens_val, tokens_test = self.transformer.generate_embeddings(
            train_df=train_df,
            val_df=val_df,
            test_df=test_df,
        )

        test_dataset = ReviewDataset(
            tokens_test,
            test_labels.tolist()
        )

        test_loader = DataLoader(
            dataset=test_dataset,
            batch_size=8,
            num_workers=4
        )
        model = self.transformer
        mse, mae, r2, self.predicted_review_score_values = self.transformer.test_model(
            model=model,
            test_dataloader=test_loader
        )
        print(mse, mae, r2, self.predicted_review_score_values)
        print("BERT Prediction evaluation:")
        print(f"MSE: {mse}")
        print(f"MAE: {mae}")
        print(f"R2: {r2}")

        if isinstance(self.predicted_review_score_values, torch.Tensor):
            self.predicted_review_score_values = np.array([
            score.detach().cpu().item() if hasattr(score, "detach") else float(score)
            for score in self.predicted_review_score_values
            ])
        self.predicted_prices = self.regressor.predict_prices(
            review_scores=self.predicted_review_score_values,
        )
        self.predicted_prices = np.array(self.predicted_prices).flatten()
        target_prices = (
            test_df["price"]
                .replace('[\$,]', '', regex=True)
                .astype(float)
                .tolist()
        )

        mse = mean_squared_error(target_prices, self.predicted_prices)
        mae = mean_absolute_error(target_prices, self.predicted_prices)
        r2 = r2_score(target_prices, self.predicted_prices)

        print("Price Prediction Evaluation:")
        print(f"  MSE: {mse:.2f}")
        print(f"  MAE: {mae:.2f}")
        print(f"  R²: {r2:.2f}")

        return self.predicted_prices, self.predicted_review_score_values

## Visualization of the results from BERT

In [ ]:
from decepticons import decepticons, ReviewDataset
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple Silicon GPU)")
else:
    device = torch.device("cpu")
    print("Using CPU")

dec = decepticons()
dec.load_weights()
dec.eval()
dec.to(device)

# Split data
train_df, val_df, test_df = dec.split_data()

# Create binary labels from price
train_labels, val_labels, test_labels = dec.discretize_labels(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df
)

# Tokenize
tokens_train, tokens_val, _ = dec.generate_embeddings(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df
)

_, _, tokens_test = dec.generate_embeddings(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df
)

# Dataset & DataLoader
train_dataset = ReviewDataset(tokens_train, train_labels.tolist())
val_dataset = ReviewDataset(tokens_val, val_labels.tolist())
dataset = ReviewDataset(tokens_test, test_labels.tolist())

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=8, num_workers=4)
test_loader = DataLoader(dataset, batch_size=8, num_workers=4)

mse, mae, r2, raw_preds = dec.test_model(dec, test_loader)

# Predicted values from model
preds = raw_preds.detach().cpu().numpy().flatten()

# True values
true = np.array(test_labels)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=true, y=preds, alpha=0.6)
plt.plot([min(true), max(true)], [min(true), max(true)], color='red', linestyle='--')
plt.xlabel("Actual Review Score (Discretized)")
plt.ylabel("Predicted Review Score")
plt.title("Predicted vs. Actual Review Scores")
plt.grid(True)
plt.tight_layout()
plt.show()


In [43]:
import os
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score
from transformers import BertTokenizer, BertModel
import pandas as pd
import torch
from transformers import BertTokenizerFast, AutoModel
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader
from torch.nn import CrossEntropyLoss, MSELoss
from pathlib import Path

"""
General TODO list:
 - Write weights to file and read in [x]
 - Train more than one epoch [x] -> done by runnign the script multiple times
 - Combine with regression head to determination [] -> Pipeline: BERT encodes semantic meaning of the comment ->
    Predicts review score -> translates sentiment into expected price
 - Test Bert with custom reviews []
 - Visualize []
"""
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple Silicon GPU)")
else:
    device = torch.device("cpu")
    print("Using CPU")

class decepticons(nn.Module):

    """
    Load Transformer Model
        - Choose a pre-trained Transformer Model
        - Load tokenizer and model from Hugging Face Transformers
    """
    def __init__(self):
        super(decepticons, self).__init__()
        self.bert = AutoModel.from_pretrained('bert-base-uncased')
        for param in self.bert.parameters():
            param.requires_grad = False

        self.dropout = nn.Dropout(0.2)
        self.rectified_linear_unit = nn.ReLU()
        self.linear_layer_1 = nn.Linear(768, 512)
        self.linear_layer_2 = nn.Linear(512, 512)
        self.output_layer = nn.Linear(512, 1)  # Final scalar output
        self.listings = pd.read_csv("./data/listings.csv")
        self.reviews = pd.read_csv("./data/reviews.csv")
        self.decepticons_weights_path = Path("./checkpoints/decepticons.pt")
        self.test_mode = True
        self.prediction_mode = False


    """
     Prepare Text Features
        - Load data and preprocess data as for transformers
        - @source https://mccormickml.com/2021/06/29/combining-categorical-numerical-features-with-bert/
    """
    def prepare_features(self):
        # Normalize column names
        self.listings.columns = self.listings.columns.str.strip().str.lower()
        self.reviews.columns = self.reviews.columns.str.strip().str.lower()

        # Merge listings and reviews
        merged = pd.merge(
            self.listings,
            self.reviews,
            left_on="id",
            right_on="listing_id",
            how="left"
        )

        # Rename for clarity
        merged = merged.rename(columns={
            "id_x": "id",
            "id_y": "review_id"
        })

        # Select relevant columns
        df = merged[["id", "price", "review_scores_value","comments"]]

        # Drop rows where any of the selected columns have NaN
        df = df.dropna(subset=["id", "price", "review_scores_value","comments"])

        return df

    """
    Running a forward pass through the model
        - first providing the inputs to the bert model
        - Second running it through the
    """
    def forward(self, sent_id, mask):
        outputs = self.bert(sent_id, attention_mask=mask, return_dict=True)
        cls_embedding = outputs.pooler_output  # shape [batch_size, 768]

        x = self.linear_layer_1(cls_embedding)  # 768 → 512
        x = self.rectified_linear_unit(x)
        x = self.dropout(x)
        x = self.linear_layer_2(x)  # 512 → 512
        x = self.rectified_linear_unit(x)
        x = self.dropout(x)

        out = self.output_layer(x)  # 512 → 1
        return out.squeeze(-1)  # [batch_size]

    """
    Splits the data into train, validation and test sets.
    Returns:
        train_df, val_df, test_df: DataFrames split accordingly
    """
    def split_data(self, test_size=0.2, val_size=0.1, random_state=42):

        df = self.prepare_features()

        # First split off test set
        train_val_df, test_df = train_test_split(
            df, test_size=test_size, random_state=random_state, shuffle=True
        )

        # Then split train_val into train and val
        val_relative_size = val_size / (1 - test_size)  # adjust val size relative to train_val

        train_df, val_df = train_test_split(
            train_val_df, test_size=val_relative_size, random_state=random_state, shuffle=True
        )

        return train_df, val_df, test_df

    """
    Generate Text Embeddings
        - Tokenize the text data
        - Pass tokenized input through the model to get embeddings
        @Source https://www.geeksforgeeks.org/nlp/fine-tuning-bert-model-for-sentiment-analysis/
    """
    def generate_embeddings(self, train_df=None, test_df=None, val_df=None, batch_size=16):
        # load tokenizer
        tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        pad_len = int(train_df["comments"].str.len().mean())

        tokens_train = tokenizer.batch_encode_plus(
            train_df["comments"].tolist(),
            max_length=pad_len,
            padding=True,
            truncation=True
        )

        tokens_test = tokenizer.batch_encode_plus(
            test_df["comments"].tolist(),
            max_length=pad_len,
            padding=True,
            truncation=True
        )

        tokens_val = tokenizer.batch_encode_plus(
            val_df["comments"].tolist(),
            max_length=pad_len,
            padding=True,
            truncation=True
        )

        return tokens_train, tokens_val, tokens_test

    """
    Discretize Classification labels as of review_score_value
    """
    def discretize_labels(self, train_df=None, val_df=None, test_df=None):
        train_labels = pd.qcut(train_df['review_scores_value'], q=5, labels=[1, 2, 3, 4, 5]).astype(int)
        val_labels = pd.qcut(val_df['review_scores_value'], q=5, labels=[1, 2, 3, 4, 5]).astype(int)
        test_labels = pd.qcut(test_df['review_scores_value'], q=5, labels=[1, 2, 3, 4, 5]).astype(int)
        return train_labels, val_labels, test_labels

    """
    Evaluate and Tune
        - Evaluate on test Data set
    """
    def evaluate_model(self, model, val_dataloader):
        print("\nEvaluating...")
        model.to(device)
        model.eval()

        total_loss = 0
        total_preds = []

        for step, batch in enumerate(val_dataloader):
            if step % 50 == 0 and not step == 0:
                print('  Batch {:>5,}  of  {:>5,}.'.format(step, len(val_dataloader)))

            sent_id = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            with torch.no_grad():
                preds = model(sent_id, mask)
                loss = MSELoss()(preds, labels)
                total_loss += loss.item()
                total_preds.append(preds)

        avg_loss = total_loss / len(val_dataloader)
        total_preds = torch.cat(total_preds, dim=0)

        return avg_loss, total_preds

    """
    Export Model & Inference Script
        - Save trained model weights
    """
    def export_weights(self):
        try:
            torch.save(self.to('cpu').state_dict(), self.decepticons_weights_path)
            self.to(device)  # Move back to original device
            print(f"Saved weights to {self.decepticons_weights_path}")
        except Exception as e:
            print(f"Failed to save weights: {e}")

    def load_weights(self):
        print("Loading weights...")
        print(f"Loading weights from {self.decepticons_weights_path}")
        try:
            state_dict = torch.load(self.decepticons_weights_path, map_location=device)
            self.load_state_dict(state_dict)
            print("Weights loaded successfully!")
        except Exception as e:
            print(f"Failed to load weights: {e}")
            # Optionally: Delete corrupted file and retrain

    """
    Train the Model
        - Define loss (e.g., MSELoss) and optimizer (e.g., Adam)
        - Set up training loop with validation
        - @source https://machinelearningmastery.com/adam-optimization-algorithm-for-deep-learning/
    """
    def optimize_model(self, model, train_dataloader):
        model.to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
        model.train()
        total_loss = 0
        total_preds = []

        for step, batch in enumerate(train_dataloader):
            if step % 50 == 0 and not step == 0:
                print('  Batch {:>5,}  of  {:>5,}.'.format(step, len(train_dataloader)))

            sent_id = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            model.zero_grad()
            preds = model(sent_id, mask)
            loss_fn = MSELoss()
            loss = loss_fn(preds, labels)
            total_loss += loss.item()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
            optimizer.step()
            total_preds.append(preds.detach())

        avg_loss = total_loss / len(train_dataloader)
        total_preds = torch.cat(total_preds, dim=0)

        return avg_loss, total_preds

    @staticmethod
    def test_model(model, test_dataloader):
        print("\nTesting...")
        model.to(device)
        model.eval()

        all_preds = []
        all_labels = []

        for batch in test_dataloader:
            sent_id = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            with torch.no_grad():
                preds = model(sent_id, mask)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        mse = mean_squared_error(all_labels, all_preds)
        mae = mean_absolute_error(all_labels, all_preds)
        r2 = r2_score(all_labels, all_preds)

        return mse, mae, r2, all_preds

class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = {k: torch.tensor(v) for k, v in encodings.items()}
        self.labels = torch.tensor(labels, dtype=torch.float)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx]
        }

if __name__ == "__main__":

        dec = decepticons()
        dec.load_weights()
        dec.eval()
        dec.to(device)

        # Split data
        train_df, val_df, test_df = dec.split_data()

        # Create binary labels from price
        train_labels, val_labels, test_labels = dec.discretize_labels(
            train_df=train_df,
            val_df=val_df,
            test_df=test_df
        )

        # Tokenize
        tokens_train, tokens_val, _ = dec.generate_embeddings(
            train_df=train_df,
            val_df=val_df,
            test_df=test_df
        )

        _, _, tokens_test = dec.generate_embeddings(
            train_df=train_df,
            val_df=val_df,
            test_df=test_df
        )

        # Dataset & DataLoader
        train_dataset = ReviewDataset(tokens_train, train_labels.tolist())
        val_dataset = ReviewDataset(tokens_val, val_labels.tolist())
        dataset = ReviewDataset(tokens_test, test_labels.tolist())

        train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=4)
        val_loader = DataLoader(val_dataset, batch_size=8, num_workers=4)
        test_loader = DataLoader(dataset, batch_size=8, num_workers=4)


        mse, mae, r2, raw_preds = dec.test_model(dec, test_loader)

        # Predicted values from model
        preds = raw_preds.detach().cpu().numpy().flatten()

        # True values
        true = np.array(test_labels)

        plt.figure(figsize=(8, 6))
        sns.scatterplot(x=true, y=preds, alpha=0.6)
        plt.plot([min(true), max(true)], [min(true), max(true)], color='red', linestyle='--')
        plt.xlabel("Actual Review Score (Discretized)")
        plt.ylabel("Predicted Review Score")
        plt.title("Predicted vs. Actual Review Scores")
        plt.grid(True)
        plt.tight_layout()
        plt.show()








Using MPS (Apple Silicon GPU)
Loading weights...
Loading weights from checkpoints/decepticons.pt
Weights loaded successfully!

Testing...


Traceback (most recent call last):
  File "<string>", line 1, in <module>
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/multiprocessing/spawn.py", line 116, in spawn_main
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/multiprocessing/spawn.py", line 116, in spawn_main
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/multiprocessing/spawn.py", line 126, in _main
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    exitcode = _main(fd, parent

RuntimeError: DataLoader worker (pid(s) 92572) exited unexpectedly